# 12_03.여러 페이지 크롤링(BeautifulSoup)



## 1.기본 package 설정


In [1]:
## 1.기본
import numpy as np  # numpy 패키지 가져오기
import pandas as pd # pandas 패키지 가져오기
import matplotlib.pyplot as plt # 시각화 패키지 가져오기

## 2.크롤링
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm

In [ ]:
# 네이버에서 접속 제한 풀기
# req = requests.get(news_url, headers={'User-agent': 'Mozilla/5.0'})
# soup = BeautifulSoup(req.text, "lxml")  # html에 대하여 접근할 수 있도록

In [2]:
# 네이버에서 접속 제한 풀기
# User-Agent 확인: https://www.useragentstring.com/
header = ({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"})

## 2.검색어 설정

In [8]:
search_addr = (f"https://search.naver.com/search.naver?where=news&sm=tab_pge"
                 "&query=%EB%B9%85%EB%8D%B0%EC%9D%B4%ED%84%B0&sort=0&photo=0&field=0&pd=0&ds=&de=&cluster_rank=30"
                 "&mynews=0&office_type=0&office_section_code=0&news_office_checked=&nso=so:r,p:all,a:all&start=")

## 3.URL주소 검색

In [9]:
start, last, step = 1, 50, 10
news_url = []

for i in range(start, last, step):
    import requests   # 접속할때 마다 실행해야 함
    search_url = search_addr + str(i)
    requests = requests.get(search_url, headers = header)
    bs = BeautifulSoup(requests.text)
    url = bs.select("div.news_wrap.api_ani_send > div > div.news_info > div.info_group > a:nth-child(3)")

    for url in tqdm(url):
        news_url.append(url["href"])

news_url

100%|██████████| 5/5 [00:00<00:00, 7628.78it/s]


['https://n.news.naver.com/mnews/article/003/0012008608?sid=102',
 'https://n.news.naver.com/mnews/article/421/0006964221?sid=102',
 'https://n.news.naver.com/mnews/article/215/0001117029?sid=101',
 'https://n.news.naver.com/mnews/article/082/0001225130?sid=102',
 'https://n.news.naver.com/mnews/article/081/0003381684?sid=100',
 'https://n.news.naver.com/mnews/article/018/0005542413?sid=101',
 'https://n.news.naver.com/mnews/article/003/0012001698?sid=102',
 'https://n.news.naver.com/mnews/article/056/0011537362?sid=101',
 'https://n.news.naver.com/mnews/article/030/0003121969?sid=101',
 'https://n.news.naver.com/mnews/article/092/0002300552?sid=103',
 'https://n.news.naver.com/mnews/article/421/0006957521?sid=105',
 'https://n.news.naver.com/mnews/article/003/0012005629?sid=105',
 'https://n.news.naver.com/mnews/article/003/0011987882?sid=101',
 'https://n.news.naver.com/mnews/article/421/0006962608?sid=102',
 'https://n.news.naver.com/mnews/article/008/0004919391?sid=101',
 'https://

## 4.뉴스 저장


In [6]:
news_no = []
news_title = []
news_body = []
news_press = []

for i, url in tqdm(enumerate(news_url)):
    import requests                                       # 접속할때 마다 실행해야 함
    requests = requests.get(url, headers = header)            # 개별 페이지 header
    news_html = BeautifulSoup(requests.text,"html.parser")
    news_html

    # 글번호
    no = i + 1
    news_no.append(no)

    # 제목
    title = news_html.select_one("#title_area > span").text
    news_title.append(title)

    # 본문
    body = news_html.select_one("#dic_area").text
    news_body.append(body)

    # 신문사
    press = news_html.select_one("img.media_end_head_top_logo_img.light_type")["title"]
    news_press.append(press)


22it [00:11,  1.98it/s]


## 5.테이블로 저장

In [8]:
news_df = pd.DataFrame({'번호': news_no, '제목':news_title,'본문':news_body,'출판사':news_press})
news_df

,번호,제목,본문,출판사
0,1,"서울신용보증재단, 소상공인 마이데이터 서비스 시작",\n데이터 형태 행정서류 활용한 편리한 서류제출 지원\n\n\n\n[서울=뉴시스]소...,뉴시스
1,2,"강릉시, '빅데이터'로 관광트렌드 대응",\n강릉 관광 빅데이터 분석·실태조사 수립 사업 완료보고회당일치기 여행 증가·로컬미...,뉴스1
2,3,"""제주도 관광 연구""…신한카드, 데이터 결합 사업 추진",\n\n\n\n\n신한카드는 민간 데이터전문기관으로서 가명 정보를 활용한 첫 번째 ...,한국경제TV
3,4,"한국수산자원공단, 행안부 공공 빅데이터 분석 지원사업 선정","\n\n\n\n\n한국수산자원공단(이하 수산공단, 이사장 이춘우) 수산종자산업진흥센...",부산일보
4,5,"노원구, 청년 빅데이터 전문가 키운다… 데이터 사이언스 무료 교육",\n이달 19일 개강… 2개월간 데이터 지식 활용법 강의오승록 구청장 “관련 분야 ...,서울신문
5,6,"빅밸류, AI로 금융권 점포 이전·폐쇄, 마케팅 전략 수립",\n인공지능과 공간 데이터 결합한 AI LOBIG데이터 기반 지역 마케팅전략 수립 ...,이데일리
6,7,질병청-심평원 건강정보 빅데이터 구축 위해 '맞손',\n양 기관 보유 데이터 공유 MOU\n\n\n\n[서울=뉴시스]충북 오송 소재 질...,뉴시스
7,8,[성공예감] 투자자에게 꼭 필요한 핵심 경제지표를 알자 – 김두언 (빅데이터 이코노...,\n\n\n\n\n====================================...,KBS
8,9,"교육부, 직업계고 교원 대상 빅테크 기업 방문 연수 실시",\n\n\n\n\n직업계고 교원 미래 직업교육 역량 강화 프로그램에 참여한 교원들이...,전자신문
9,10,"中 지리, 세계 첫 '자동차업 AI 빅모델' 출시 예고",\n하반기 '인허 L6'에도 AI 기능 탑재중국 자동차 기업 지리자동차가 자동차 산...,ZDNet Korea


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# news_df.to_csv('/content/drive/MyDrive/Colab Notebooks/news_df.csv')

In [ ]:
# news_df.to_csv('/content/drive/MyDrive/Colab Notebooks/news_df.csv')
# news_df.head()